# Neural Architecture Search - search for an edge-friendly network

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khanhnd61-vr/modelopt-101/blob/main/examples/04_nas.ipynb)

**Goal.** Define a network search space and compare a mixed random/evolutionary strategy
against random-only search under the same number of unique candidate evaluations.

This completed lab covers the core NAS workflow and all experiments from the original
`Takeaways & things to try` section:

1. expand widths with `48` and `64`;
2. evaluate two deployment objectives on the same search results;
3. re-evaluate the top three candidates with a stronger proxy;
4. compare mixed search with random-only at the exact same realized budget;
5. fully train only the hand-designed baseline and the selected NAS winner.

Search uses validation accuracy. The test set is reserved for the final fully trained models.

**Runtime:** set `Runtime -> Change runtime type -> GPU`, then `Runtime -> Run all`.

## 1. Setup

On Colab `torch`/`torchvision` are pre-installed, so nothing to `pip install`.

In [ ]:
import io, time, copy, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as T

GLOBAL_SEED = 0
RANDOM_ONLY_SEED = 314
FINAL_TRAIN_SEED = 2026

torch.manual_seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
random.seed(GLOBAL_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# --- full training and low-fidelity search ---
EPOCHS = 10
SEARCH_EPOCHS = 2
SEARCH_SUBSET = 10_000
BATCH_SIZE = 128
LR = 0.05

# --- mixed random/evolutionary search ---
N_RANDOM = 8
EVO_GENS = 2
EVO_CHILDREN = 4

# --- stronger proxy used only to re-rank the top three ---
TOP_K_RECHECK = 3
HIGH_PROXY_EPOCHS = 4
HIGH_PROXY_SUBSET = 20_000

# Alternative edge objective: smallest model within this accuracy distance from the best.
NEAR_BEST_TOLERANCE_PP = 0.5

## 2. Data - train, validation, search subsets, and test

The 60,000 original training images are deterministically split into 50,000 training and
10,000 validation examples. Search candidates are ranked on validation accuracy. The test set
is touched only after full training of the baseline and selected winner.

The low-fidelity proxy uses 10,000 training images; top-three rechecking uses 20,000.

In [ ]:
mean = (0.1307,)
std = (0.3081,)

train_tf = T.Compose([
    T.RandomCrop(28, padding=2),
    T.ToTensor(),
    T.Normalize(mean, std),
])
eval_tf = T.Compose([T.ToTensor(), T.Normalize(mean, std)])

full_train_aug = torchvision.datasets.MNIST(
    "./data", train=True, download=True, transform=train_tf
)
full_train_eval = torchvision.datasets.MNIST(
    "./data", train=True, download=True, transform=eval_tf
)
test_set = torchvision.datasets.MNIST(
    "./data", train=False, download=True, transform=eval_tf
)

split_generator = torch.Generator().manual_seed(GLOBAL_SEED)
all_indices = torch.randperm(
    len(full_train_aug), generator=split_generator
).tolist()
train_indices = all_indices[:50_000]
val_indices = all_indices[50_000:]

train_set = Subset(full_train_aug, train_indices)
val_set = Subset(full_train_eval, val_indices)
search_set = Subset(full_train_aug, train_indices[:SEARCH_SUBSET])
high_proxy_set = Subset(full_train_aug, train_indices[:HIGH_PROXY_SUBSET])

val_loader = DataLoader(val_set, batch_size=256, shuffle=False, num_workers=2)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2)

def make_train_loader(dataset, seed):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        generator=generator,
    )

print(
    f"{len(train_set)} train / {len(val_set)} validation / {len(test_set)} test\n"
    f"low proxy={len(search_set)} images, high proxy={len(high_proxy_set)} images"
)

## 3. Expanded search space

An architecture has three genes: depth, first-stage width, and kernel size. Widths `48` and
`64` extend the original space without adding a new gene or complicating mutation logic.

Channel width doubles after each stage and is capped at 128.

In [ ]:
CHOICES = {
    "depth": [2, 3, 4],
    "width": [8, 12, 16, 24, 32, 48, 64],
    "kernel": [3, 5],
}


def build_model(arch, num_classes=10):
    layers = []
    input_channels = 1
    kernel = arch["kernel"]

    for stage in range(arch["depth"]):
        output_channels = min(arch["width"] * (2 ** stage), 128)
        layers.extend([
            nn.Conv2d(
                input_channels,
                output_channels,
                kernel,
                padding=kernel // 2,
                bias=False,
            ),
            nn.BatchNorm2d(output_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        ])
        input_channels = output_channels

    layers.extend([
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Linear(input_channels, num_classes),
    ])
    return nn.Sequential(*layers)


def random_arch():
    return {gene: random.choice(values) for gene, values in CHOICES.items()}


def arch_key(arch):
    return tuple(arch[gene] for gene in CHOICES)


def arch_str(arch):
    return f"d{arch['depth']}-w{arch['width']}-k{arch['kernel']}"


def arch_seed(arch):
    """Stable seed: the same architecture gets the same initialization in every proxy."""
    return (
        GLOBAL_SEED
        + arch["depth"] * 10_000
        + arch["width"] * 100
        + arch["kernel"]
    )


space_size = int(np.prod([len(values) for values in CHOICES.values()]))
print(f"expanded search space: {space_size} architectures")
print("example:", arch_str(random_arch()))

## 4. Training, evaluation, and proxy utilities

Every occurrence of an architecture uses a stable seed. This makes its low- and
high-fidelity evaluations reproducible and reduces ranking noise caused by initialization.

In [ ]:
def count_params(model):
    return sum(parameter.numel() for parameter in model.parameters())


def train(model, loader, epochs, lr=LR):
    model.to(device).train()
    optimizer = torch.optim.SGD(
        model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = F.cross_entropy(model(x), y)
            loss.backward()
            optimizer.step()
        scheduler.step()
    return model


@torch.no_grad()
def evaluate(model, loader):
    was_training = model.training
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total += y.numel()
    if was_training:
        model.train()
    return 100.0 * correct / total


@torch.no_grad()
def latency_ms(model, n=100):
    model_cpu = copy.deepcopy(model).to("cpu").eval()
    x = torch.randn(1, 1, 28, 28)
    for _ in range(10):
        model_cpu(x)
    start = time.perf_counter()
    for _ in range(n):
        model_cpu(x)
    return (time.perf_counter() - start) / n * 1000.0


def model_size_mb(model):
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    return buf.getbuffer().nbytes / 1e6


def proxy_eval(arch, dataset, epochs):
    """Train one architecture from scratch and return validation accuracy and cost."""
    seed = arch_seed(arch)
    torch.manual_seed(seed)
    model = build_model(arch)
    loader = make_train_loader(dataset, seed)
    model = train(model, loader, epochs=epochs)
    return {
        "acc": evaluate(model, val_loader),
        "size": model_size_mb(model),
        "params": count_params(model),
    }

## 5. Main search - random exploration

The mixed strategy first evaluates eight unique random architectures with the cheap proxy.
Every accepted candidate is stored in `results`; duplicate architectures cost nothing.

In [ ]:
results = []
seen = set()


def consider(arch, source):
    key = arch_key(arch)
    if key in seen:
        return False

    seen.add(key)
    metrics = proxy_eval(arch, search_set, SEARCH_EPOCHS)
    row = {
        "arch": dict(arch),
        "source": source,
        **metrics,
    }
    results.append(row)
    print(
        f"  {arch_str(arch):<13} val_acc={row['acc']:5.2f}% "
        f"size={row['size']:5.2f}MB [{source}]"
    )
    return True


print("mixed search - random phase:")
while sum(row["source"] == "random" for row in results) < N_RANDOM:
    consider(random_arch(), "random")

## 6. Main search - evolutionary exploitation

Each generation selects the two highest-accuracy candidates evaluated so far. Mutation changes
exactly one gene to another allowed value. The loop counts only unique accepted children, so
the realized mixed budget is exactly `N_RANDOM + EVO_GENS * EVO_CHILDREN`.

In [ ]:
def mutate(arch):
    child = dict(arch)
    gene = random.choice(list(CHOICES))
    alternatives = [value for value in CHOICES[gene] if value != arch[gene]]
    child[gene] = random.choice(alternatives)
    return child


print("mixed search - evolutionary phase:")
for generation in range(EVO_GENS):
    parents = sorted(
        results,
        key=lambda row: row["acc"],
        reverse=True,
    )[:2]
    print(
        f" generation {generation + 1}: "
        f"parents={[arch_str(parent['arch']) for parent in parents]}"
    )

    accepted = 0
    attempts = 0
    while accepted < EVO_CHILDREN:
        parent = random.choice(parents)["arch"]
        accepted += int(consider(mutate(parent), "evolved"))
        attempts += 1

        # A fallback guarantees progress if local mutations repeatedly hit seen candidates.
        if attempts > 100 and accepted < EVO_CHILDREN:
            accepted += int(consider(random_arch(), "evolved"))

mixed_budget = len(results)
expected_budget = N_RANDOM + EVO_GENS * EVO_CHILDREN
assert mixed_budget == expected_budget
print(f"\nrealized mixed-search budget: {mixed_budget} unique candidates")

## 7. Main-search Pareto front

A candidate is Pareto-optimal when no other evaluated architecture is both at least as
accurate and no larger, with one comparison strictly better.

In [ ]:
import matplotlib.pyplot as plt


def pareto_front(rows):
    front = []
    for row in rows:
        dominated = any(
            other is not row
            and other["acc"] >= row["acc"]
            and other["size"] <= row["size"]
            and (
                other["acc"] > row["acc"]
                or other["size"] < row["size"]
            )
            for other in rows
        )
        if not dominated:
            front.append(row)
    return sorted(front, key=lambda row: row["size"])


front = pareto_front(results)
print("Mixed-search Pareto front:")
for row in front:
    print(
        f"  {arch_str(row['arch']):<13} val_acc={row['acc']:5.2f}% "
        f"size={row['size']:5.2f}MB source={row['source']}"
    )

fig, axis = plt.subplots(figsize=(8, 5.5))
for source, color in [("random", "#777777"), ("evolved", "#d1495b")]:
    points = [row for row in results if row["source"] == source]
    axis.scatter(
        [row["size"] for row in points],
        [row["acc"] for row in points],
        color=color,
        label=source,
        s=65,
    )
axis.plot(
    [row["size"] for row in front],
    [row["acc"] for row in front],
    "--o",
    color="#2a9d8f",
    label="Pareto front",
)
axis.set_xlabel("model size (MB) - lower is better")
axis.set_ylabel("low-proxy validation accuracy (%)")
axis.set_title("Mixed search: accuracy versus cost")
axis.grid(alpha=0.25)
axis.legend()
plt.tight_layout()
plt.show()

## 8. Two objectives on the same results

The original objective picks the most accurate candidate no larger than a hand-designed
baseline. The edge-oriented alternative picks the smallest candidate within 0.5 percentage
points of the best observed proxy accuracy. No additional training is needed.

In [ ]:
BASELINE_ARCH = {"depth": 3, "width": 32, "kernel": 3}
baseline_size = model_size_mb(build_model(BASELINE_ARCH))

# Objective A: highest proxy accuracy within the baseline-size budget.
within_baseline_budget = [
    row for row in results if row["size"] <= baseline_size
]
winner_budget = max(
    within_baseline_budget or results,
    key=lambda row: row["acc"],
)

# Objective B: smallest model within 0.5pp of the best proxy accuracy.
best_proxy_accuracy = max(row["acc"] for row in results)
near_best = [
    row for row in results
    if row["acc"] >= best_proxy_accuracy - NEAR_BEST_TOLERANCE_PP
]
winner_small = min(
    near_best,
    key=lambda row: (row["size"], -row["acc"]),
)

# The edge-oriented objective chooses the final architecture to train fully.
chosen_row = winner_small
chosen = chosen_row["arch"]

print(f"baseline architecture: {arch_str(BASELINE_ARCH)} ({baseline_size:.2f}MB)")
print(
    "objective A - best accuracy within baseline size: "
    f"{arch_str(winner_budget['arch'])}, "
    f"acc={winner_budget['acc']:.2f}%, size={winner_budget['size']:.2f}MB"
)
print(
    f"objective B - smallest within {NEAR_BEST_TOLERANCE_PP:.1f}pp of best: "
    f"{arch_str(winner_small['arch'])}, "
    f"acc={winner_small['acc']:.2f}%, size={winner_small['size']:.2f}MB"
)
print(f"selected for full training: {arch_str(chosen)}")

## 9. Stronger-proxy recheck of the top three

The top three low-proxy candidates are retrained from scratch with the same architecture seed,
but twice the data and twice the epochs. This experiment checks whether the cheap proxy's
ranking is stable. It is a diagnostic only; the two objectives above remain based on equal
low-fidelity scores for every candidate.

In [ ]:
top_low_proxy = sorted(
    results,
    key=lambda row: row["acc"],
    reverse=True,
)[:TOP_K_RECHECK]

proxy_recheck = []
for low_rank, row in enumerate(top_low_proxy, start=1):
    high_metrics = proxy_eval(
        row["arch"],
        high_proxy_set,
        HIGH_PROXY_EPOCHS,
    )
    proxy_recheck.append({
        "arch": row["arch"],
        "low_rank": low_rank,
        "low_acc": row["acc"],
        "high_acc": high_metrics["acc"],
    })

high_order = sorted(
    proxy_recheck,
    key=lambda row: row["high_acc"],
    reverse=True,
)
for high_rank, row in enumerate(high_order, start=1):
    row["high_rank"] = high_rank

print(f"{'architecture':<15}{'low acc':>11}{'low rank':>11}{'high acc':>12}{'high rank':>12}")
print("-" * 61)
for row in sorted(proxy_recheck, key=lambda item: item["low_rank"]):
    print(
        f'{arch_str(row["arch"]):<15}{row["low_acc"]:>10.2f}%'
        f'{row["low_rank"]:>11}{row["high_acc"]:>11.2f}%{row["high_rank"]:>12}'
    )

low_order_keys = [arch_key(row["arch"]) for row in proxy_recheck]
high_order_keys = [arch_key(row["arch"]) for row in high_order]
print(f"\nTop-three order preserved: {low_order_keys == high_order_keys}")

## 10. Random-only search at the exact same budget

The mixed search realized `len(results)` unique evaluations. Random-only now receives exactly
that many unique evaluations in the same expanded search space, using a separate random seed.
This is the fair comparison requested by the original lab.

In [ ]:
saved_random_state = random.getstate()
random.seed(RANDOM_ONLY_SEED)

random_only_results = []
random_only_seen = set()

print("random-only search:")
while len(random_only_results) < mixed_budget:
    arch = random_arch()
    key = arch_key(arch)
    if key in random_only_seen:
        continue
    random_only_seen.add(key)

    metrics = proxy_eval(arch, search_set, SEARCH_EPOCHS)
    row = {"arch": dict(arch), "source": "random-only", **metrics}
    random_only_results.append(row)
    print(
        f"  {arch_str(arch):<13} val_acc={row['acc']:5.2f}% "
        f"size={row['size']:5.2f}MB"
    )

random.setstate(saved_random_state)
assert len(random_only_results) == mixed_budget

random_only_front = pareto_front(random_only_results)

def strategy_summary(name, rows):
    ordered = sorted(rows, key=lambda row: row["acc"], reverse=True)
    return {
        "name": name,
        "evaluations": len(rows),
        "best_acc": ordered[0]["acc"],
        "top3_mean": np.mean([row["acc"] for row in ordered[:3]]),
        "pareto_count": len(pareto_front(rows)),
        "best_arch": ordered[0]["arch"],
    }

strategy_results = [
    strategy_summary("mixed", results),
    strategy_summary("random-only", random_only_results),
]

print(f"\n{'strategy':<14}{'evals':>8}{'best acc':>12}{'top-3 mean':>14}{'Pareto pts':>12}")
print("-" * 60)
for row in strategy_results:
    print(
        f'{row["name"]:<14}{row["evaluations"]:>8}'
        f'{row["best_acc"]:>11.2f}%{row["top3_mean"]:>13.2f}%'
        f'{row["pareto_count"]:>12}'
    )

## 11. Search diagnostics

The plots below show high-fidelity top-three re-ranking, equal-budget strategy quality, and
the two objective choices on the shared mixed-search result set.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5))

# Low- versus high-fidelity ranking for the same top-three architectures.
labels = [arch_str(row["arch"]) for row in proxy_recheck]
x = np.arange(len(labels))
bar_width = 0.36
axes[0].bar(
    x - bar_width / 2,
    [row["low_acc"] for row in proxy_recheck],
    width=bar_width,
    label=f"low: {SEARCH_EPOCHS}ep/{SEARCH_SUBSET}",
    color="#457b9d",
)
axes[0].bar(
    x + bar_width / 2,
    [row["high_acc"] for row in proxy_recheck],
    width=bar_width,
    label=f"high: {HIGH_PROXY_EPOCHS}ep/{HIGH_PROXY_SUBSET}",
    color="#2a9d8f",
)
axes[0].set_xticks(x, labels, rotation=15)
axes[0].set_ylabel("validation accuracy (%)")
axes[0].set_title("Top-three proxy recheck")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.2)

# Equal-budget random-only versus mixed strategy.
strategy_names = [row["name"] for row in strategy_results]
x_strategy = np.arange(len(strategy_names))
axes[1].bar(
    x_strategy - bar_width / 2,
    [row["best_acc"] for row in strategy_results],
    width=bar_width,
    label="best",
    color="#d1495b",
)
axes[1].bar(
    x_strategy + bar_width / 2,
    [row["top3_mean"] for row in strategy_results],
    width=bar_width,
    label="top-3 mean",
    color="#f4a261",
)
axes[1].set_xticks(x_strategy, strategy_names)
axes[1].set_ylabel("low-proxy validation accuracy (%)")
axes[1].set_title(f"Strategy comparison ({mixed_budget} evaluations each)")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.2)

# Same candidates, different deployment objectives.
axes[2].scatter(
    [row["size"] for row in results],
    [row["acc"] for row in results],
    color="#999999",
    s=45,
    label="mixed candidates",
)
for row, color, label in [
    (winner_budget, "#d1495b", "objective A"),
    (winner_small, "#2a9d8f", "objective B / selected"),
]:
    axes[2].scatter(row["size"], row["acc"], color=color, s=130, label=label)
    axes[2].annotate(
        arch_str(row["arch"]),
        (row["size"], row["acc"]),
        xytext=(6, 6),
        textcoords="offset points",
    )
axes[2].axvline(
    baseline_size,
    linestyle="--",
    color="#555555",
    label="baseline size",
)
axes[2].axhline(
    best_proxy_accuracy - NEAR_BEST_TOLERANCE_PP,
    linestyle=":",
    color="#7b2cbf",
    label="best - 0.5pp",
)
axes[2].set_xlabel("model size (MB)")
axes[2].set_ylabel("low-proxy validation accuracy (%)")
axes[2].set_title("Two objectives, one result set")
axes[2].legend(fontsize=8)
axes[2].grid(alpha=0.2)

plt.tight_layout()
plt.show()

## 12. Full training of the baseline and selected winner

Only two models receive the full ten-epoch, 50,000-image budget: the hand-designed baseline
and the architecture selected by objective B. Both use the same training seed and data order.

In [ ]:
print("training hand-designed baseline ...")
torch.manual_seed(FINAL_TRAIN_SEED)
baseline = build_model(BASELINE_ARCH)
baseline = train(
    baseline,
    make_train_loader(train_set, FINAL_TRAIN_SEED),
    epochs=EPOCHS,
)
baseline_val_acc = evaluate(baseline, val_loader)
base_acc = evaluate(baseline, test_loader)
print(f"  validation={baseline_val_acc:.2f}% test={base_acc:.2f}%")

print("training selected NAS winner ...")
torch.manual_seed(FINAL_TRAIN_SEED)
best = build_model(chosen)
best = train(
    best,
    make_train_loader(train_set, FINAL_TRAIN_SEED),
    epochs=EPOCHS,
)
best_val_acc = evaluate(best, val_loader)
best_acc = evaluate(best, test_loader)
print(f"  validation={best_val_acc:.2f}% test={best_acc:.2f}%")

## 13. Final deployment comparison

The final table uses test accuracy only after architecture selection and full training. Model
size, parameters, and batch-size-one CPU latency quantify deployment cost.

In [ ]:
final_specs = [
    (f"Baseline {arch_str(BASELINE_ARCH)}", BASELINE_ARCH, baseline, base_acc),
    (f"NAS {arch_str(chosen)}", chosen, best, best_acc),
]

final_results = []
for name, arch, model, accuracy in final_specs:
    final_results.append({
        "name": name,
        "arch": arch,
        "params": count_params(model),
        "size_mb": model_size_mb(model),
        "accuracy": accuracy,
        "latency_ms": latency_ms(model),
    })

print(f"{'model':<25}{'params':>13}{'size':>11}{'test acc':>12}{'CPU latency':>15}")
print("-" * 76)
for row in final_results:
    print(
        f'{row["name"]:<25}{row["params"]:>13,}'
        f'{row["size_mb"]:>9.2f}MB{row["accuracy"]:>11.2f}%'
        f'{row["latency_ms"]:>13.2f}ms'
    )

names = [row["name"] for row in final_results]
colors = ["#4c4c4c", "#2a9d8f"]
metrics = [
    ("accuracy", "Test accuracy", "%"),
    ("size_mb", "Model size", "MB"),
    ("latency_ms", "CPU latency / image", "ms"),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for axis, (key, title, unit) in zip(axes, metrics):
    values = [row[key] for row in final_results]
    bars = axis.bar(names, values, color=colors)
    axis.set_title(title)
    axis.tick_params(axis="x", rotation=12)
    axis.grid(axis="y", alpha=0.2)
    if key == "accuracy":
        axis.set_ylim(max(0, min(values) - 3), min(100, max(values) + 1))
    for bar, value in zip(bars, values):
        axis.text(
            bar.get_x() + bar.get_width() / 2,
            value,
            f"{value:.2f}{unit}",
            ha="center",
            va="bottom",
        )

plt.tight_layout()
plt.show()

baseline_result, nas_result = final_results
print("\nFINAL EVALUATION")
print("=" * 78)
print(
    f'Selected objective: smallest architecture within '
    f'{NEAR_BEST_TOLERANCE_PP:.1f}pp of the best low-proxy validation accuracy.'
)
print(
    f'The NAS winner is {baseline_result["size_mb"] / nas_result["size_mb"]:.2f}x '
    f'smaller and {baseline_result["latency_ms"] / nas_result["latency_ms"]:.2f}x '
    f'faster than the hand-designed baseline.'
)
print(
    f'Its fully trained test-accuracy change is '
    f'{nas_result["accuracy"] - baseline_result["accuracy"]:+.2f}pp.'
)
print(
    f'Mixed and random-only search each evaluated {mixed_budget} unique candidates; '
    f'their best low-proxy accuracies were '
    f'{strategy_results[0]["best_acc"]:.2f}% and {strategy_results[1]["best_acc"]:.2f}%.'
)

## 14. Takeaways

- **Search space:** adding widths 48 and 64 increases the space from 30 to 42 architectures.
- **Search strategy:** mixed search and random-only are compared using the exact same number
  of unique proxy evaluations; one run is illustrative rather than a statistical proof.
- **Objective:** the best-accuracy-under-budget and smallest-near-best rules can select
  different architectures from exactly the same candidate table.
- **Proxy fidelity:** top-three rechecking reveals whether a 2-epoch/10k-image ranking remains
  stable with a 4-epoch/20k-image proxy.
- **Final evidence:** proxy accuracy is used for search; only the fully trained baseline and
  selected winner support the final test-accuracy, size, and latency comparison.

NAS finds a promising architecture before other model-optimization techniques are applied.
The selected network can subsequently be distilled, pruned, and quantized for deployment.